# 12 · 模型设置工具 — LLM / VLM / Embedding / OCR / ASR (+确定性内核)

可编辑模型注册表 `config/models.yaml`; UI(`GET /`) 6 张卡片改端点/模型/启用 + 测试连接; 后端 `GET/POST /v1/models/config` + `POST /v1/models/probe`。改动即生效(Planner/funasr/router 从此读取)。
铁律①: `deterministic` 锁定, 不可禁用/不可改成 LLM。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from bootstrap import build_controller, status
from services.config import load_settings, load_policy
print('trunk root:', ROOT)

In [ ]:
from services import model_config as mc
cfg = mc.load()
print('注册表模型:', list(cfg['models'].keys()))
for k, m in cfg['models'].items():
    print('  %-14s %-10s %-34s %s %s' % (k, m.get('role'), m.get('endpoint'), m.get('model'), '🔒' if m.get('locked') else ''))

## 真实探活 (在线/离线 + 时延)

In [ ]:
probe = mc.probe_all(timeout=3.0)
for k, v in probe.items():
    print('  %-16s online=%-5s %sms  %s' % (k, v.get('online'), v.get('latency_ms'), '' if v.get('online') else (v.get('error') or v.get('skipped') or '')[:40]))

## 校验: 缺端点/非法scheme/禁用确定性内核 → 拦截

In [ ]:
import copy
bad = copy.deepcopy(cfg); bad['models']['llm']['endpoint'] = ''
print('缺 endpoint:', mc.validate(bad))
bad2 = copy.deepcopy(cfg); bad2['models']['asr']['endpoint'] = '127.0.0.1:8089'
print('非法 scheme:', mc.validate(bad2))
bad3 = copy.deepcopy(cfg); bad3['models']['deterministic']['enabled'] = False
print('禁用确定性内核:', mc.validate(bad3))

## UI + API 端点 (TestClient, 无需起服务)

In [ ]:
from fastapi.testclient import TestClient
from services.api_server import app
c = TestClient(app)
html = c.get('/').text
print('GET / :', len(html), 'bytes | 含模型设置:', '模型设置' in html)
d = c.get('/v1/models/config').json()
print('config models:', list(d['config']['models'].keys()), '| probe keys:', list(d['probe'].keys())[:3], '...')
p = c.post('/v1/models/probe', data={'key':'deterministic'}).json()
print('probe deterministic online:', p['deterministic']['online'])
bad = copy.deepcopy(d['config']); bad['models']['deterministic']['enabled']=False
print('禁用确定性内核 HTTP:', c.post('/v1/models/config', json=bad).status_code, '(应400)')